# Levelised cost of green ammonia by cell: global results

This notebook shows the results of a global campaign of the green-lory plant model as maps, bar
charts, histograms, tables and supply curves. Each run of the campaign designs and costs an islanded
plant, which makes ammonia from wind and PV, in every one-degree cell, and gives the levelised cost of
ammonia (LCOA) of each cell's plant. The notebook reads the merged result tables of the global stage
of `arc/submit_lory_sequence.sh` (`merged/global_run_results.csv`, checked by the merge and QA job,
`arc/qa.py`), copied from ARC to `results/campaigns/`. `run_store.csv` gives the folder of each run,
and `model/results.py` reads the tables by run id and draws the figures. The notebook builds and runs
nothing.

In [ ]:
import importlib

import matplotlib.pyplot as plt
import pandas as pd

import model.results as h

importlib.reload(h)
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", None)  # the rule descriptions in full

## Runs and cases of the campaign

Every run solves the same plant in each cell: wind turbines and fixed PV supply an electrolyser,
batteries, hydrogen and ammonia storage and a Haber-Bosch synthesis loop, with no grid connection,
over the 8,760 hours of 2019 in ERA5 weather. The plant is designed to make 1 Mt/yr of ammonia, and
its LCOA (`lcoa_eur_per_t`) is its annualised cost divided by its annual output, in EUR per tonne.
How much ammonia a cell can make at that cost is limited by the land the cell gives to the plant, all
the land that the land rules leave open to wind and PV (section "How the land rules set the production
potential"), and is the cell's production potential
(`scaled_design_max_gridless_onshore_ammonia_capacity_t`, t/yr).

A land run covers the cells with suitable land. A sea run (a run id with `_offshore`) puts the same
plant at sea, with wind turbines and no PV, in every cell that is at least 1 % sea, on all of the
cell's sea outside protected areas and at the costs of the cell's water depth (section "Plants
at sea"). A coastal cell so holds a land plant and a sea plant, each on its own area.

The four cases differ only in the costs that depend on the location:

- `base`: a 5 % cost of capital everywhere, capital costs without location factors, and water at a
  uniform price of 2 EUR/m³ (2020 euros);
- `site`: capital costs multiplied by the location factors of the run's site-cost table
  (`inputs/site_costs/`), which are remoteness (equipment by the distance to a port, installation by
  the travel time to a town), cold and slope on land, and remoteness and cold at sea; water is priced
  at its source plus a pipeline from the nearest source;
- `finance`: the cost of capital of the cell's country, from the reduced rates of Ameli et al. (2021),
  the series that Verschuur et al. (2024) used, and 5.1 %, the series' highest rate, in international
  waters;
- `all`: the site costs of `site` with the cost of capital of `finance`.

`base` leaves out every cost that depends on the location, so remote, cold or frozen sites can look
cheap in `base` and `finance`; `all`, with every one of these costs, gives the most complete picture.
The low-carbon rates of Ameli et al., from 2.4 % in Japan to 11.8 % in Africa, are a sensitivity of
`finance` and `all` (the `..._amelilc_...` runs of `runs.yaml`).
The land runs (`..._land100oct_...`) are derived from solved runs that gave each plant 20 % of the
same land (`..._land20oct_...`): the land enters only after the solve, so they keep those runs'
designs and LCOA and multiply the production potential by five (`python -m model.suppliers derive`).
The sea runs (`..._offshore_mpa`) are derived in the same way from runs that gave each plant 20 % of
its cell's sea, protected areas included, and also multiply the potential by the unprotected share of
the cell's water.
The table lists the runs with the options `run_store.csv` records for each (the tokens are defined in
`GLOSSARY.md`), its status and whether its results are in the local copies.

In [ ]:
# Land runs of the four cases: run ids from runs.yaml (`green-lory list` shows the others). These take all
# the land that the land rules leave; their parents, "..._land20oct_..." in place of "land100oct", take 20 %.
RUNS = {
    "base": "gl_dea2050_wacc5_bflat_wflat_land100oct_fixed",
    "site": "gl_dea2050_wacc5_bspat_wspat_land100oct_fixed",
    "finance": "gl_dea2050_amelired_bflat_wflat_land100oct_fixed",
    "all": "gl_dea2050_amelired_bspat_wspat_land100oct_fixed",
}
# Sea runs of the four cases, on all of each cell's sea outside protected areas; their parents,
# "..._land20oct_fixed_offshore", take 20 % of each cell's sea, protected areas included
SEA_RUNS = {
    "base": "gl_dea2050_wacc5_bflat_wflat_land100oct_fixed_offshore_mpa",
    "site": "gl_dea2050_wacc5_bspat_wspat_land100oct_fixed_offshore_mpa",
    "finance": "gl_dea2050_amelired_bflat_wflat_land100oct_fixed_offshore_mpa",
    "all": "gl_dea2050_amelired_bspat_wspat_land100oct_fixed_offshore_mpa",
}
h.runs_table({**RUNS, **{f"{case} at sea": run_id for case, run_id in SEA_RUNS.items()}})

In [ ]:
for run_id in [*RUNS.values(), *SEA_RUNS.values()]:
    if not h.output_dir(run_id).is_dir():
        print(f"{run_id} has no local copy; copy it from ARC with\n  {h.copy_command(run_id)}")
LAND = {case: h.load_surface(run_id) for case, run_id in RUNS.items()}
SEA = {case: h.load_surface(run_id) for case, run_id in SEA_RUNS.items()}

## Location factors of the cost

The cases differ by three location factors, mapped here as the runs used them, from their result
tables, each in its own colour. The first map shows `build_cost_multiplier`, the factor by which the
location raises the capital cost of the land plants of `site` (it is 1 in `base` and `finance`). It
is close to 1 in most of the world and rises with remoteness and cold: the median cell has 1.3, the
central Sahara and the Tibetan Plateau, far from towns, have 2 to 3, and a tenth of the cells have
more than 3.6, most of them in the north of Russia and Canada, in Alaska and in Greenland.

In [ ]:
LIMITS = (1, 4)  # ends of the colour scale; larger factors take the end colour
factor = LAND["site"]["build_cost_multiplier"]
h.plot_maps({"": LAND["site"]}, "build_cost_multiplier", title="Location factor on the capital cost of the land plants: site",
            label="Factor on capital cost (-)", limits=LIMITS, ramp=h.RAMPS["orange"],
            notes={"": h.summary_note(factor, decimals=2)})
plt.show()

The chart shows where the build factor comes from (`h.plot_build_factors`; `model/site_costs.py`,
`site_costs` in `inputs/parameters.yaml`). The remote factor rises with the logarithm of the travel
time d, in days, from the cell to the nearest city of 20,000 people: 1 + 1.127 log2(1 + d) on the
installation part of the capital cost and 1 + 0.676 log2(1 + d) on its equipment part (solid lines).
Both pass through their calibration point (the crosses), Shamattawa, Manitoba, 0.85 days from such a
city, where a Canadian cost manual gives 1.99 for construction and 1.60 for transport, rounded to 2
and 1.6. The cold factor
multiplies both by 1.25 at 60 degrees of latitude or more, or at 3,000 m or more (dashed lines). The
points are the location factors of two Australian cost reviews, which the rule did not use: GHD's
installation factors and Aurecon's installation and equipment factors, at each site's travel time.
The bars below count the land plants of `site` that can make 1 Mt/yr by travel time, those under the
cold factor darker; the dotted lines mark the remotest Australian site and the calibration point.

In [ ]:
SITE_RUN = RUNS["site"]  # the run whose site-cost table and plants are shown
h.plot_build_factors(LAND["site"], h.site_cost_table(SITE_RUN),
                     title="Remote and cold factors on the capital cost against travel time: site")
plt.show()

Near cities the rule agrees with the Australian factors: its installation factor lies 0.04 below
GHD's on average and 0.19 below Aurecon's, which are derived as the residual of an overall location
factor, and its equipment factor lies 0.01 above Aurecon's. The Australian sites end 6.7 hours from a
city, and 62 % of the 13,156 plants lie within that; 18 % lie between it and the calibration point,
0.85 days, and 20 % beyond, up to 6.3 days, where the rule has no data and its factors keep rising: on
installation 2.13 at one day, 2.79 at two and 3.62 at four, and on equipment 1.68, 2.07 and 2.57. The
cold factor raises each by a quarter, and 68 % of the plants beyond the calibration point are under
it, so these two extrapolations together set the cost of most remote plants.

The second map shows the cost of capital of `finance` (`interest_rate_overall`), which is set by
country and is the same for every technology of a plant. A sea plant takes the rate of the country
whose exclusive economic zone holds its cell, and 5.1 % in international waters. The finance table
gives one rate to each of the 16 regions for which Ameli et al. estimate the cost of capital; the
table under the maps lists them (`finance_region` of `h.cell_regions`), with their land cells. The
rate is 5.1 % in 11 of the regions, which hold 12,697 of the 16,885 land cells, and lower in the
other five: 1.8 % in Japan, 4.0 % in Western Europe, 4.3 % in the United Kingdom, 4.4 % in the
United States and 4.6 % in Canada. `finance` so differs little from `base`, at 5 % everywhere.

In [ ]:
LIMITS = (1, 6)  # ends of the colour scale in per cent; (2, 12) for the low-carbon rates of the sensitivity
finance = {f"{site} plants": plants.assign(interest_rate_pct=100 * plants["interest_rate_overall"])
           for site, plants in (("Land", LAND["finance"]), ("Sea", SEA["finance"]))}
h.plot_maps(finance, "interest_rate_pct", title="Cost of capital of the plants: finance", label="Cost of capital (%)",
            limits=LIMITS, ramp=h.RAMPS["aqua"])
plt.show()
regions = finance["Land plants"].merge(h.cell_regions(), on=["latitude", "longitude"]).groupby("finance_region")
regions.agg(cells=("interest_rate_pct", "size"), cost_of_capital_pct=("interest_rate_pct", "median")).round(1).sort_values("cost_of_capital_pct")

The third map shows the water cost of the land plants of `site` per tonne of ammonia
(`water_cost_eur_per_t`): the price of water at its source plus a pipeline from the nearest source,
times the plant's water use. It is highest in the deserts, far from rivers and lakes, but small
against the LCOA: a median of 6.6 EUR/t and at most 79 EUR/t, against 3.2 EUR/t at the uniform price
of `base` and `finance`. Sea plants desalinate sea water at the source price, with no pipeline.

In [ ]:
LIMITS = None  # ends of the colour scale in EUR/t, or None for the 2nd and 98th percentiles
water = LAND["site"]["water_cost_eur_per_t"]
h.plot_maps({"": LAND["site"]}, "water_cost_eur_per_t", title="Water cost of the land plants: site",
            label="Water cost (EUR/t of ammonia)", limits=LIMITS, ramp=h.RAMPS["blue"],
            notes={"": h.summary_note(water, "EUR/t", decimals=1)})
plt.show()

## Levelised cost of ammonia by case

The maps show the four cases one after the other, on one colour scale. Each cell shows the cheaper of
its land and sea plants among those that can make at least 1 Mt/yr (`h.lowest_feasible_plant`), the
plants that count towards the supply curves and the network model's suppliers; `SITES` shows the land
or the sea plants alone. The colours are those of viridis, reversed so that yellow is the cheapest,
on a linear scale from 250 to 2,000 EUR/t; plants above 2,000 EUR/t are grey, and cells where no plant
can make 1 Mt/yr are light grey. The box in each panel gives the number of land and sea plants shown
and their median LCOA. The lines under the maps give, for each case, the median cost of the land
plants north of 60 degrees N and the share of them above 2,000 EUR/t; the sea plants up to 2,000
EUR/t and those of them south of 60 degrees S; and the cells that hold a land plant and a sea plant
that can both make 1 Mt/yr, with those of them where the sea plant is the cheaper.

In [ ]:
CASES = list(RUNS)  # the cases shown, one panel each, from "base", "site", "finance" and "all"
SITES = "both"  # "both" (each cell's cheaper plant), "land" or "sea"
MINIMUM_MT = 1.0  # the plants shown can make at least this (Mt/yr); 0 shows every plant
LIMITS = (250, 2000)  # ends of the colour scale in EUR/t; plants above the upper end are grey
shown = {case: h.select_plants(LAND[case], SEA[case], SITES, MINIMUM_MT) for case in CASES}
cells = h.select_plants(LAND[CASES[0]], SEA[CASES[0]], SITES, 0.0)
h.plot_maps(shown, "lcoa_eur_per_t", label="Levelised cost of ammonia (EUR/t)", limits=LIMITS, ramp=h.LCOA_COLOURMAP,
            title=f"Levelised cost of ammonia by case: {h.SITES[SITES]}" + (f" that can make {MINIMUM_MT:g} Mt/yr" if MINIMUM_MT else ""),
            above=f"above {LIMITS[1]:,} EUR/t", background=cells, background_label=f"no plant that makes {MINIMUM_MT:g} Mt/yr",
            notes={case: h.site_note(frame) for case, frame in shown.items()})
plt.show()
for case, frame in shown.items():
    north = frame.loc[(frame["site"] == "land") & (frame["latitude"] > 60), "lcoa_eur_per_t"]
    sea = frame[(frame["site"] == "sea") & (frame["lcoa_eur_per_t"] <= LIMITS[1])]
    two = h.compare_surfaces(*(plants[plants[h.CAPACITY_COLUMN] >= MINIMUM_MT * 1e6] for plants in (SEA[case], LAND[case])))
    print(f"{case}: land plants north of 60 degrees N {north.median():,.0f} EUR/t, {(north > LIMITS[1]).mean():.0%} above {LIMITS[1]:,} EUR/t; "
          f"{len(sea):,} sea plants up to {LIMITS[1]:,} EUR/t, {(sea['latitude'] <= -60).sum():,} of them south of 60 degrees S; "
          f"{len(two):,} cells with two plants, in {(two['lcoa_change_pct'] < 0).sum():,} of which the sea plant is the cheaper")

In `base`, the 13,198 land plants that can make 1 Mt/yr cost a median 332 EUR/t and are almost all
yellow, the cheap end of the scale; only north of 60 degrees N, where the median plant costs 526
EUR/t, do they turn green. The sea plants cost a median 1,063 EUR/t; 32,209 of them cost up to
2,000 EUR/t, the cheapest, in green, in the windy belts of the Southern Ocean, the North Atlantic
and the North Pacific, and 4,139 of these lie south of 60 degrees S, off Antarctica, since `base`
has no cost for cold or remoteness. `site` adds these costs: the median land plant rises to 412
EUR/t, the central Sahara and the interior of Australia turn green, the plants north of 60 degrees N
rise to a median 1,479 EUR/t, in blue and purple, and only 9,949 sea plants stay up to 2,000 EUR/t,
along the coasts and in parts of the Southern Ocean, because the remoteness factors of a sea plant
count the distance to the nearest port. `finance` looks almost the same as `base`, since its cost
of capital is 5.1 % in most of the world: the median land plant costs 332 EUR/t in both, the plants
north of 60 degrees N 518 EUR/t, and 32,168 sea plants stay up to 2,000 EUR/t. `all` looks almost
the same as `site` for the same reason: the median land plant costs 410 EUR/t, a fifth of the plants
north of 60 degrees N cost more than 2,000 EUR/t, and 9,996 sea plants stay up to 2,000 EUR/t, 45
of them south of 60 degrees S.

About 1,500 cells hold a land plant and a sea plant that can both make 1 Mt/yr. The land plant is the
cheaper in all of them in `base` and `finance`, and in all but two in `site` and `all`, on the Gydan
peninsula of the Russian Arctic (71 degrees N, 77 and 78 degrees E), where the cold and remoteness
factors make both plants cost more than 1,000 EUR/t. A coastal cell so shows its sea plant almost only
where its land cannot make 1 Mt/yr. Light grey marks the
3,687 of the 16,885 land cells whose plant cannot make 1 Mt/yr even on all the land the rules leave
(the boxes of the maps in section "How the land rules set the production potential" give the
counts). In 2,370 of them, the land-cover shares leave too little land, as in the boreal forests of
Russia, Canada and Scandinavia and the tropical forests of the Amazon, the Congo Basin, South-east
Asia and New Guinea; in 744, protected areas and steep slopes do; and 573, such as small islands,
have too little land of any kind. At sea, light grey marks 2,592 cells in `base`: 2,348 of them have at
least half of their water in protected areas (section "Plants at sea"), and most of the other 244 are
coastal cells with little sea.

The table lists the ten cheapest land plants of one case that can make at least 1 Mt/yr, with the
cell's mean elevation (`elevation_m`; below zero where much of the cell is sea), wind's share of the
plant's generating capacity and the cell's production potential.

In [ ]:
CASE = "base"  # "base", "site", "finance" or "all"
MINIMUM_MT = 1.0  # the table lists plants that can make at least this (Mt/yr); 0 lists every plant
h.cheapest_cells(LAND[CASE], n=10, minimum_mt_per_year=MINIMUM_MT)

In `base`, the cheapest land plants of all, at 225 to 256 EUR/t, are on small islands, most of them
sub-Antarctic (the Crozet, Prince Edward, Kerguelen, Heard, Macquarie, Campbell and Auckland Islands)
or off southern Chile, where wind supplies most of the plant's power; but none of them has the land to
make 1 Mt/yr, the largest making 0.24 Mt/yr (`MINIMUM_MT = 0` lists them), and `base` has no cost for
their remoteness. The cheapest plants that can make 1 Mt/yr are on the high plateau of the Andes, the
Puna and the Altiplano of Argentina, Chile and Bolivia, at 3,800 to 4,900 m, where PV alone powers the
plant at 257 to 260 EUR/t; each cell can make 62 to 199 Mt/yr. `base` has no cost for building at altitude,
in the cold or far from towns, so the high plateaus rank as its cheapest sites; `site` and `all` move
them down. The ten cheapest plants of `all` are in Japan, at a cost of capital of 1.8 %, and in
Hawaii, at 275 to 286 EUR/t, but each can make only 1.0 to 2.3 Mt/yr (`CASE = "all"` lists them).

The next table gives the headline figures of every case, for the land and for the sea plants, and
the cost of the last tonne at two global production volumes: the LCOA of the plant that brings the
cumulative potential of the cheapest plants to that volume (`h.cost_at_supply`). Only plants that can
make at least 1 Mt/yr count towards a volume, as for the network model's suppliers.

In [ ]:
VOLUMES = (500, 2000)  # global production volumes (Mt/yr) at which the table gives the cost of the last tonne
h.summary_table(LAND, SEA, volumes=VOLUMES)

## What the location factors add to the cost, by region

The chart splits the mean LCOA of `all` by UN sub-region (M49, the United Nations Statistics
Division's standard regions, `un_sub_region` of `h.cell_regions`): the cost of `base`, and on it what
each location factor adds (`h.cost_factors`): construction and water, from `site`; the cost of
capital, from `finance`; and the part that the two give together, since the cost of capital of `all`
applies to the capital cost that its site costs have raised. The means are over the land plants that
can make 1 Mt/yr in every case, with the number of plants of each sub-region in its label; unlike
medians, means add up, so each bar ends at its sub-region's mean LCOA, marked by a black tick. A part
below zero, such as finance where the cost of capital is below 5 %, is drawn left of zero. M49 puts all
of Russia in Eastern Europe, and Greenland and Canada in Northern America with the United States. The
line under the chart gives the mean of each part over all the plants, and the table the means of each
sub-region in EUR/t.

In [ ]:
MINIMUM_MT = 1.0  # the plants included can make at least this (Mt/yr) in every case; 0 includes every land plant
BY = "un_sub_region"  # "un_sub_region", "un_region" (M49) or "finance_region" (the 16 regions of the finance table)
factors = h.cost_factors(LAND["base"], LAND["site"], LAND["finance"], LAND["all"])
factors = factors[factors["potential_mt_per_year"] >= MINIMUM_MT]
h.plot_region_costs(factors, ["construction", "water", "finance", "interaction"], by=BY,
                    title=f"Mean levelised cost of ammonia by region: all, {len(factors):,} land plants")
plt.show()
print("Mean of each part (EUR/t):", ", ".join(f"{h.FACTOR_NAMES[name]} {value:,.0f}"
                                            for name, value in factors[list(h.FACTOR_NAMES)].mean().items()))
means = factors.groupby(BY)[list(h.FACTOR_NAMES)].mean()
means.assign(all=means.sum(axis=1)).sort_values("all").round(0).astype(int)

Over the 13,133 land plants, `base` costs a mean 371 EUR/t. Site costs add 255 EUR/t, nearly all of
it construction (247 EUR/t, against 7 EUR/t of water), finance takes off 2 EUR/t, and the two
together take off 2 EUR/t more, for a mean of 622 EUR/t in `all`. Construction weighs most in
Eastern Europe, which in M49 includes Russia, and in Northern America, which hold a third of the
plants (4,586), many of them in the cold and remote north of Russia, Canada and Alaska: construction
adds 655 and 467 EUR/t to their means, which makes them the dearest sub-regions, at 1,173 and 866
EUR/t.

`all` is cheapest in Southern Europe (337 EUR/t), and next in Southern, South-eastern and Western
Asia, Latin America and the Caribbean and Western Europe, at 363 to 383 EUR/t. After the two
northern sub-regions, construction adds most in Northern Europe (172 EUR/t), Northern Africa (150
EUR/t) and Melanesia (144 EUR/t), and then in Australia and New Zealand, Eastern Asia and Central
Asia (97 to 112 EUR/t), where many plants are far from towns or ports. Finance changes little: at
5.1 %, the rate of most countries, it adds at most 4 EUR/t to a sub-region's mean, and the lower
rates of Europe, the United States and Canada take 17 to 38 EUR/t off the means of Southern,
Western and Northern Europe and Northern America. Northern Europe has the dearest plants of `base`
(535 EUR/t), since most of them lie far north, in Scandinavia, Finland and Iceland.

## Plants at sea

A sea plant's base costs depend on the mean water depth of its cell (`depth_class` in the run's
site-cost table): fixed foundations up to 70 m, floating foundations to 1,000 m, and deep water
beyond. The wind turbine costs about as much as on land, while its foundation, array cables and
development cost 5.5 times as much with fixed foundations and 9 times as much with floating ones; in
deep water the whole turbine costs 3 times as much. The rest of the plant costs 1.72, 2 and 3 times as
much as on land in the three classes. The map shows the classes, with the land cells in grey. The
table gives the median LCOA of the sea plants by depth class for every case; the lines above it
count the sea cells in international waters (no `eez_iso3`) and those with water in protected areas,
give the cheapest sea plant south of 60 degrees S in each case and count the sea plants of each case
that are cheaper than its median land plant.

In [ ]:
depth = h.depth_classes(SEA_RUNS["base"])
h.plot_maps({"": depth}, "depth_class", title="Depth class of the sea plants", label="Depth class",
            categories=dict(zip(h.DEPTH_CLASSES.values(), h.DEPTH_COLOURS)), background=LAND["base"], background_label="land cells")
plt.show()
print(f"Sea cells in international waters: {depth['eez_iso3'].isna().sum():,} of {len(depth):,}")
print("Protected water:", h.protected_water_note(h.protected_areas(SEA["base"])).replace("\n", "; "))
print("Cheapest sea plant south of 60 degrees S:",
      ", ".join(f"{case} {SEA[case].loc[SEA[case]['latitude'] <= -60, 'lcoa_eur_per_t'].min():,.0f} EUR/t" for case in RUNS))
print("Sea plants cheaper than the case's median land plant:",
      ", ".join(f"{case} {(SEA[case]['lcoa_eur_per_t'] < LAND[case]['lcoa_eur_per_t'].median()).sum():,}" for case in RUNS))
h.median_by(SEA, depth[["latitude", "longitude", "depth_class"]])

The table lists the ten cheapest sea plants of one case, with the country of the exclusive economic
zone that holds each cell (`eez_iso3`, empty in international waters).

In [ ]:
CASE = "base"  # "base", "site", "finance" or "all"
h.cheapest_cells(SEA[CASE].merge(depth, on=["latitude", "longitude"]), n=10, extra=["eez_iso3", "depth_class"])

In `base`, a sea plant costs 1,069 EUR/t at the median cell, three times as much as a land plant:
746 EUR/t with fixed foundations, 811 EUR/t with floating ones and 1,123 EUR/t in deep water, which
covers 33,136 of the 40,197 sea cells. No sea plant of `base` is cheaper than the median land plant.
The cheapest, at 385 to 411 EUR/t, lie off southern Chile, around the Kerguelen Islands and off the
coast of Antarctica; the cells around the Kerguelen Islands, however, lie wholly in the marine reserve of
the French Southern Lands, so their plants can make nothing (the last column of the table). The model has no data on sea ice, so frozen sea counts as sea, and the cold rule
is a factor on cost, not a limit, which only `site` and `all` apply. The Antarctic cells are therefore
cheap in `base` and `finance`, while in `site` and `all` the cold factor (1.25 beyond 60 degrees of
latitude) and the remoteness factors make the cheapest sea plant south of 60 degrees S cost 1,780 and
1,794 EUR/t.

The cost of capital changes the sea plants as little as the land plants: 24,254 of the 40,197 sea
cells are in international waters, at 5.1 %, and the median sea plant of `finance` costs 1,063 EUR/t.
Site costs raise it, because the remoteness factors of a sea plant count the distance from the cell
to the nearest port: the median sea plant of `site` costs 2,582 EUR/t, and that of `all` 2,574
EUR/t. In `all`, one sea plant is cheaper than the median land plant (433 EUR/t), off southern Chile
(54 degrees S, 73 degrees W) at 389 EUR/t; the next, at 439 to 448 EUR/t, lie off Tierra del Fuego,
in the German Bight and off Mauritania (`CASE = "all"` lists them).

A sea plant may use all of its cell's sea outside protected areas: the areas of the World Database on
Protected Areas (WDPA, February 2026) with the statuses that the land rule leaves out, in the marine
and coastal realms (`protected_water_pct` of `inputs/site_costs/protected_areas.csv`, measured on the
land cover's grid as on land). They hold some of the water of 7,467 of the 40,197 sea cells and 9.4 %
of the water of all of them, most of it in a few large areas around remote islands and in the Ross
Sea. Leaving them out changes no plant's LCOA, only the potential of the sea plants, which the share
table and the histograms below count.

## How the land rules set the production potential

A cell's production potential is what its plant can make when the plant's design, solved for 1
Mt/yr, is scaled to fill the land the cell gives it. The land table sets that land with three rules:
the share of each land-cover class open to each technology, which leaves out land that serves other
uses (cropland, forest, cities), and the exclusion of protected areas and of steep slopes. The land
runs of the four cases take all the land that the rules leave; the end of the section shows the
effect of a land share, which gives the plant only part of that land. The maps, supply curves and
histograms keep only the plants that can make 1 Mt/yr. The first table lists what
each rule does, with the wind spacing and PV density that turn the land into capacity
(`h.land_rule_list`, from `inputs/parameters.yaml`); the second gives the share of each land-cover
class open to wind and to PV (`h.land_cover_shares`); the third, the median share of a cell's land
that each rule keeps, by UN sub-region from the most land open to wind to the least
(`h.land_share_medians`).

In [ ]:
h.land_rule_list(LAND["base"])

In [ ]:
h.land_cover_shares()

In [ ]:
BY = "un_sub_region"  # "un_sub_region" or "un_region" (UN M49), or "finance_region" (the 16 regions of the finance table)
h.land_share_medians(LAND["base"], by=BY)

The maps apply the rules one after another to the land plants of `base`
(`h.potential_by_land_rule`), starting from every km² of each cell's land open to wind and PV; the
third panel is the run's own potential, and the fourth takes 20 % of its land, the sensitivity. The
run applies land after the solve, so each plant's design scales to the land of each step without a
new solve. The colour scale runs from 1 to 20 Mt/yr: larger potentials take the darkest colour,
cells whose plant cannot make 1 Mt/yr are light grey, and the box in each panel gives the potential
of all the cells and the number that can make 1 Mt/yr. The table under the maps gives, by band of
latitude, the medians of what sets a plant's potential on all of its cell's land
(`h.land_by_latitude`): the cell's land, the land the plant needs per Mt/yr, the PV density, wind's
share of the generating capacity and the potential.

In [ ]:
LIMITS = (1, 20)  # ends of the colour scale in Mt/yr: cells below the lower end are grey, larger ones take the darkest colour
SHARES = [0.2]  # land shares of the extra panels, as fractions of the land that the rules leave
potential = h.potential_by_land_rule(LAND["base"])
h.plot_land_rule_maps(potential, limits=LIMITS, shares=SHARES,
                      title="Production potential of each land plant of base as the land rules apply")
plt.show()

In [ ]:
EDGES = (-90, -23.5, 23.5, 60, 90)  # edges of the bands of latitude in degrees, south to north
h.land_by_latitude(LAND["base"], potential, edges=EDGES)

On all of each cell's land, 16,312 of the 16,885 land cells could hold a plant of 1 Mt/yr, with
1,475 Gt/yr in all. The cells north of 60 degrees N are paler, for two reasons. A one-degree cell
holds less land there, a median 4,528 km² against 11,691 km² in the tropics, since the meridians
converge towards the pole. And a plant there needs ten times as much land per Mt/yr, a median 563
km² against 55 km² in the tropics: under the low winter sun, rows of PV modules must stand far apart
so that they do not shade each other, at a median 14 MW/km² against 100 MW/km², and the plants take
a median 38 % of their generating capacity from wind, which needs 200 km² per GW. A cell north of 60
degrees N can so make a median 7.2 Mt/yr on all its land, against 203 Mt/yr in the tropics.

The land-cover shares cut the potential most, to 506 Gt/yr. They close forests, wetlands, snow and
ice, and nearly all cropland, so that the boreal forests of Russia, Canada and Scandinavia, the
tropical forests and the farmland of Europe, India and eastern China fall to a few Mt/yr or below
1 Mt/yr, while the deserts of Northern Africa and the Arabian Peninsula, open in full as barren land,
keep nearly all of their potential. Protected areas and steep slopes take a further 64 Gt/yr, which
leaves 442 Gt/yr and 13,198 cells that can make 1 Mt/yr: the potential of the runs. Scotland, which
MODIS classes mostly as savanna and grassland, open to wind at 20 %, keeps cells that can make 1
Mt/yr. A share of 20 % of the land divides every cell's potential by five, to 88 Gt/yr, and leaves
7,961 cells that can make 1 Mt/yr, none of them in Scotland.

A land share gives the plant only part of the land that the rules leave and keeps the rest for other
users of wind and PV power. Two global studies set such limits as shares of a region's whole area,
without land-cover classes: a model of the global electricity system allows PV on 6 % and wind on 4 %
(Bogdanov et al., 2019), and a study of green ammonia allows 10 % for each (Fasihi et al., 2021).
Salmon and Bañares-Alcántara (2022) and Verschuur et al. (2024) used land-cover classes, protected
areas and slopes as here, and then gave ammonia 2 % of the land that remained, its share of global
energy demand: the energy used to make ammonia for fertiliser, and the shipping fuel that ammonia
would replace, are each about 2 % of that demand. With all the land that the rules leave, the median
cell opens 15 % of its land to wind and to PV (the last two columns of the table of median shares),
as much as the limits of Fasihi et al. The same share applies at sea, to the sea outside protected
areas. The first table gives its effect on the results of `base` and `all`
(`h.share_sensitivity`): for each share of the land and sea that the rules leave, the land and the sea
plants that can make 1 Mt/yr, their potential, and the cost of the last tonne at the two production
volumes of the summary table. The second gives, by UN sub-region, the land plants of `base` that can
make 1 Mt/yr with all the land and with a share of 20 %, and the percentage of them that the share
keeps (`h.plants_by_share`).

In [ ]:
SHARES = [0.02, 0.05, 0.1, 0.2, 0.4, 1.0]  # shares to compare, as fractions of the land and sea that the rules leave
h.share_sensitivity({case: LAND[case] for case in ("base", "all")}, SEA, SHARES, volumes=VOLUMES)

In [ ]:
SHARES = [1.0, 0.2]  # land shares compared by region: the percentage kept is that of the last share against the first
h.plants_by_share(LAND["base"], SHARES, by=BY)

Potential scales with the share, so the share sets how many plants can make 1 Mt/yr: in `base`, 2,469
land plants at 2 %, 7,961 at 20 % and 13,198 with all the land, and no sea plant at 2 %, 33,891 at 20 %
and 36,807 with all the sea outside protected areas. The cells that can still make 1 Mt/yr at 20 % are
those with land to spare, where competition for land would matter least. Of the plants that can make
1 Mt/yr on all the land, 97 % still can at 20 % in Northern Africa and 85 to 93 % in Sub-Saharan
Africa, Central and Western Asia, Australia and New Zealand and Latin America and the Caribbean,
against 28 % in Northern America, 9 % in Eastern Europe and 1 to 5 % in Western and Northern Europe,
where 2 of 180 remain. The cost of the
last tonne at 2,000 Mt/yr changes little in `base`, from 278 EUR/t at 2 % to 271 EUR/t at 20 % and
261 EUR/t with all the land, because `base` has so much potential at low cost (431 Gt/yr below 400
EUR/t with all the land, from the table under the histograms in the next section) that even 2 % of it
holds that volume. `all` has about half as much (224 Gt/yr below 400 EUR/t), and its cost at 2,000
Mt/yr falls from 374 EUR/t at 2 % to 312 EUR/t at 20 % and 300 EUR/t with all the land.

## Supply curves

A supply curve sorts the plants that can make at least 1 Mt/yr from the cheapest to the dearest and
adds up their production potential (`h.supply_curve`); its height at a production volume is the cost
of the last tonne needed to make that volume. The dashed lines mark the volumes of the summary
table. The table under the figure gives, for the largest volume, the share of the potential of the
plants that make it by country ("no country" where the result table names none, as for the sea plants
and the cells of some islands, such as the Canary Islands and Cape Verde).

In [ ]:
SITES = "both"  # "land" for the land plants, "both" for the land and sea plants
XMAX = 10_000  # right end of the curves in Mt/yr; the land plants alone can make about 440,000 Mt/yr
frames = LAND if SITES == "land" else {case: pd.concat([LAND[case], SEA[case]]) for case in RUNS}
h.plot_supply_curves(frames, title=f"Supply curves of the four cases: {'land plants' if SITES == 'land' else 'land and sea plants'}",
                     xmax=XMAX, marks=VOLUMES)
plt.show()
h.supply_shares(frames, VOLUMES[-1], by="country", top=5)

The curves are nearly flat over the first 10,000 Mt/yr, because each of the cheapest cells can make
60 to 200 Mt/yr (the table of the cheapest plants above): in `base`, the cost of the last tonne rises
only from 258 EUR/t at 500 Mt/yr to 271 EUR/t at 10,000 Mt/yr. At 2,000 Mt/yr, site costs raise the
cost of the last tonne from 261 to 301 EUR/t, finance raises it to 263 EUR/t, and the two together to
300 EUR/t. Adding the sea plants changes neither volume's cost in any case (summary table).

The cases also move where the cheapest 2,000 Mt/yr comes from. In `base`, all of it comes from the
high plateau of the Andes, in Argentina (49 %), Bolivia (26 %) and Chile (25 %), and so it does in
`finance`, where the three countries' cost of capital, 5.1 %, is close to `base`'s 5 %. The cold and
remoteness factors of `site` move it to lower and less remote sites, in Chile (30 %), Egypt (17 %),
Saudi Arabia (13 %), Peru (8 %) and Namibia (7 %). In `all`, the lower cost of capital of the United
States (4.4 %) brings its sites in: 23 % of the volume is there, 21 % in Chile, 16 % in Egypt, 7 % in
Namibia and 6 % in Saudi Arabia.

The histograms show the same plants by cost band, 25 EUR/t wide, with the production potential of
each band split by the UN region (M49) of the land plants and with the sea plants as one group; the
box in each panel gives the potential above 2,000 EUR/t, which the bars leave out. A supply curve
shows how much can be made up to a cost; the histograms show where that potential lies. The table
under them gives the potential below 400 EUR/t by group, and that of all the land and all the sea
plants.

In [ ]:
EDGES = range(250, 2001, 25)  # edges of the cost bands in EUR/t; plants above the last edge go in each panel's box
BELOW = 400  # the cost (EUR/t, one of the edges) below which the table sums the potential
tables = {case: h.potential_by_cost(LAND[case], SEA[case], EDGES) for case in RUNS}
h.plot_potential_by_cost(tables, title="Production potential by levelised cost of ammonia: plants that make 1 Mt/yr")
plt.show()
h.potential_below(tables, BELOW)

In `base`, nearly all of the land's 441 Gt/yr costs less than 400 EUR/t, and 266 Gt/yr lies in the
single band from 275 to 300 EUR/t, more than half of it in Africa and most of the rest in Asia. Site
costs spread that potential over 300 to 800 EUR/t and nearly halve the potential below 400 EUR/t, to
228 Gt/yr. The cost of capital changes little: `finance` keeps 432 Gt/yr below 400 EUR/t, as `base`
does, and `all` 224 Gt/yr, close to `site`. The sea plants' 570 Gt/yr lies almost all above 550
EUR/t, and above 2,000 EUR/t for 24 Gt/yr of it in `base` and `finance` and 377 and 378 Gt/yr in
`site` and `all`.

## Base case with Salmon's weather profiles against ERA5

`base` was also run on the hourly 2019 profiles of Salmon (2022), in campaign `campaign_20261030_v1`,
with the same inputs otherwise, so the change in each cell's LCOA between the two runs comes from the
weather data alone. The map shows that change. The table gives it in bands of wind's share of the
plant's generating capacity with Salmon's profiles (`wind_share_old`, from `h.wind_share`).

In [ ]:
# base on Salmon's profiles: its execution of 30 October 2026 (run_store.csv, notes of its run)
PREVIOUS = ("results/campaigns/campaign_20261030_v1/50_campaign/gl_dea2050_wacc5_bflat_wflat_land20oct_fixed/runs/"
            "20261030T131150Z-cdbdc6788e6d3c34d291ed8234e3b14f807d54d1/global")
LIMITS = (-30, 30)  # ends of the colour scale in per cent
previous = h.read_surface(h.REPO_ROOT / PREVIOUS)
weather = h.compare_surfaces(LAND["base"], previous)
h.plot_maps({"": weather}, "lcoa_change_pct", title="Change in levelised cost of ammonia from Salmon's profiles to ERA5: base, land plants",
            label="Change in levelised cost (%)", scale="change", limits=LIMITS,
            notes={"": h.summary_note(weather["lcoa_change_pct"], "%", signed=True)})
plt.show()
print(f"Cells cheaper with ERA5: {(weather['lcoa_change_pct'] < 0).mean():.0%}")
for name, plants in (("Salmon's profiles", previous), ("ERA5", LAND["base"])):
    print(f"{name}: {(plants['wind_mw'] > 1).mean():.0%} of the plants build wind; all the plants together have "
          f"{plants['wind_mw'].sum() / 1e6:.1f} TW of wind and {plants['solar_mw'].sum() / 1e6:.1f} TW of PV")
BANDS = (0, 0.01, 0.25, 0.5, 1)  # edges of the bands of wind's share of generating capacity with Salmon's profiles
h.by_band(weather, "lcoa_change_pct", "wind_share_old", BANDS)

With ERA5, the LCOA of `base` falls by 4 % at the median cell, and 64 % of the cells are cheaper. The
change follows how much of the plant's generating capacity was wind with Salmon's profiles. Plants
with less than a quarter of their capacity in wind are 6 to 8 % cheaper at the median; plants with a
quarter to a half are 9 % dearer, and plants with more than half are 25 % dearer. On the map, the
dearer cells lie mostly in the windy north of Russia and Canada, in Kazakhstan, Scandinavia,
Patagonia and the Great Plains of the United States, while parts of eastern Siberia, Alaska and the
Yukon, and Mongolia are more than 20 % cheaper. Fewer plants build wind: the share that build any
falls from 68 % to 49 %, and all the plants together have 11.8 TW of wind instead of 17.9 TW, and
89.3 TW of PV instead of 83.4 TW.